# 02 — Data Exploration
## Greater London House Price Analysis

Exploración del dataset Silver para entender la distribución de precios, 
calidad del dato y patrones generales.

### Tabla de entrada
`workspace.default.london_silver` — 2,384,979 filas · 21 columnas

In [0]:
df_bronze = spark.table("workspace.default.london_price_paid")
df_silver = spark.table("workspace.default.london_silver")

print(f"Bronze: {df_bronze.count():,} filas")
print(f"Silver: {df_silver.count():,} filas")

## 1. Estadísticas básicas
Analizamos precio, rango de fechas y distribución por tipo de propiedad.

In [0]:
from pyspark.sql.functions import col, count, when, min, max, avg, round as spark_round

# Estadísticas de precio
print("=== ESTADÍSTICAS DE PRECIO (Silver) ===")
df_silver.select(
    spark_round(min("price"), 0).alias("precio_minimo"),
    spark_round(max("price"), 0).alias("precio_maximo"),
    spark_round(avg("price"), 0).alias("precio_medio"),
).show()

# Rango de fechas
print("=== RANGO DE FECHAS ===")
df_silver.select(
    min("date_of_transfer").alias("fecha_inicio"),
    max("date_of_transfer").alias("fecha_fin")
).show()

# Tipos de propiedad
print("=== DISTRIBUCIÓN POR TIPO DE PROPIEDAD ===")
df_silver.groupBy("property_type") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

## 2. Calidad del dato
Verificamos nulos, valores únicos en columnas categóricas y duplicados.

In [0]:
print("=== CALIDAD DEL DATO — NULOS POR COLUMNA ===")
nulos = df_silver.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_silver.columns
])
nulos.show()

print("=== VALORES INESPERADOS EN COLUMNAS CATEGÓRICAS ===")
# property_type solo debe tener D, S, T, F, O
print("property_type únicos:")
df_silver.select("property_type").distinct().show()

# duration solo debe tener F, L, U
print("duration únicos:")
df_silver.select("duration").distinct().show()

# record_status solo debe tener A
print("record_status únicos:")
df_silver.select("record_status").distinct().show()

print("=== DUPLICADOS ===")
total = df_silver.count()
distintos = df_silver.select("transaction_id").distinct().count()
print(f"Total filas: {total:,}")
print(f"Transaction IDs únicos: {distintos:,}")
print(f"Duplicados: {total - distintos:,}")

## 3. Enriquecimiento de la tabla Silver
Añadimos columnas calculadas para facilitar el análisis posterior:
- `quarter` — trimestre de la transacción
- `property_type_desc` — descripción legible del tipo de propiedad
- `price_range` — segmentación por rango de precio

In [0]:
from pyspark.sql.functions import quarter, when

# Añadir columnas calculadas útiles para el análisis
df_silver_enriquecida = df_silver \
    .withColumn("quarter", quarter(col("date_of_transfer"))) \
    .withColumn("property_type_desc", 
        when(col("property_type") == "D", "Detached")
        .when(col("property_type") == "S", "Semi-Detached")
        .when(col("property_type") == "T", "Terraced")
        .when(col("property_type") == "F", "Flat")
        .otherwise("Other")
    ) \
    .withColumn("price_range",
        when(col("price") < 250000, "Bajo (<250k)")
        .when(col("price") < 500000, "Medio (250k-500k)")
        .when(col("price") < 1000000, "Alto (500k-1M)")
        .otherwise("Premium (>1M)")
    )

print("=== DISTRIBUCIÓN POR RANGO DE PRECIO ===")
df_silver_enriquecida.groupBy("price_range") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

print("=== DISTRIBUCIÓN POR TRIMESTRE ===")
df_silver_enriquecida.groupBy("quarter") \
    .count() \
    .orderBy("quarter") \
    .show()

## 4. Guardar tabla Silver enriquecida

In [0]:
df_silver_enriquecida.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.london_silver")

print("Tabla Silver actualizada con columnas enriquecidas")
print(f"Columnas finales: {df_silver_enriquecida.columns}")